In [1]:
import torch
import onnxruntime as ort
import numpy as np

In [2]:
model = torch.hub.load('yvanyin/metric3d', 'metric3d_vit_small', pretrain=True)
model.cuda().eval()

Using cache found in /home/ukenryu/.cache/torch/hub/yvanyin_metric3d_main
/home/ukenryu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
xFormers not available
xFormers not available
xFormers not available
xFormers not available


DepthModel(
  (depth_model): DensePredModel(
    (encoder): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0): BlockChunk(
          (0-11): 12 x Block(
            (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=384, out_features=1152, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=384, out_features=384, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=384, out_features=1536, bias=True)
              (act): GELU(approximate='none')
              (fc

In [3]:
import types
funcType = types.MethodType

import torch.nn as nn
import math
def interpolate_pos_encoding_bilinear(self, x, w, h):
        previous_dtype = x.dtype
        npatch = x.shape[1] - 1
        N = self.pos_embed.shape[1] - 1
        if npatch == N and w == h:
            return self.pos_embed
        pos_embed = self.pos_embed.float()
        class_pos_embed = pos_embed[:, 0]
        patch_pos_embed = pos_embed[:, 1:]
        dim = x.shape[-1]
        w0 = w // self.patch_size
        h0 = h // self.patch_size
        # we add a small number to avoid floating point error in the interpolation
        # see discussion at https://github.com/facebookresearch/dino/issues/8
        w0, h0 = w0 + self.interpolate_offset, h0 + self.interpolate_offset

        sqrt_N = math.sqrt(N)
        sx, sy = float(w0) / sqrt_N, float(h0) / sqrt_N
        patch_pos_embed = nn.functional.interpolate(
            patch_pos_embed.reshape(1, int(sqrt_N), int(sqrt_N), dim).permute(0, 3, 1, 2),
            scale_factor=(sx, sy),
            mode="bilinear",
            antialias=self.interpolate_antialias,
        )

        assert int(w0) == patch_pos_embed.shape[-2]
        assert int(h0) == patch_pos_embed.shape[-1]
        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)
        return torch.cat((class_pos_embed.unsqueeze(0), patch_pos_embed), dim=1).to(previous_dtype)

model.depth_model.encoder.interpolate_pos_encoding = interpolate_pos_encoding_bilinear.__get__(model.depth_model.encoder, model.depth_model.encoder.__class__)

In [4]:
class Metric3DExportModel(torch.nn.Module):
    def __init__(self, meta_arch, is_export_rgb=True):
        super().__init__()
        self.meta_arch = meta_arch
        self.register_buffer('rgb_mean', torch.tensor([123.675, 116.28, 103.53]).view(1, 3, 1, 1).cuda())
        self.register_buffer('rgb_std', torch.tensor([58.395, 57.12, 57.375]).view(1, 3, 1, 1).cuda())
        self.input_size = (616, 1064)

        w_range = np.arange(0, self.input_size[1], dtype=np.float32)
        h_range = np.arange(0, self.input_size[0], dtype=np.float32)
        w_grid, h_grid = np.meshgrid(w_range, h_range) #[H, W]
        base_depth_image = np.stack([w_grid, h_grid,
                                     np.ones_like(w_grid),
                                     np.ones_like(w_grid)], axis=2)[...,np.newaxis][None] # [1, H, W, 4, 1]
        self.register_buffer("base_depth_image", torch.tensor(base_depth_image).cuda())
        self.is_export_rgb = is_export_rgb

    def normalize_image(self, image):
        image = image - self.rgb_mean
        image = image / self.rgb_std
        return image

    def depth_image_to_point_cloud_array(self, depth_image, P_inv, T, rgb_image=None, mask=None):
        """  convert depth image into color pointclouds [xyzbgr]
        depth_image: [B, 1, H, W] -> fully normalized
        K: [B, 3, 4]
        T: [B, 4, 4] -> camera to base_link / world
        rgb_image: [B, 3, H, W] -> 0-1
        mask: [B, H, W]
        """

        #[H, W, 4, 1]
        base_depth_image = torch.ones([depth_image.shape[0], depth_image.shape[2], depth_image.shape[3], 1, 1]).cuda() # [B, H, W, 1, 1]
        depth = depth_image[:, 0, :, :, None, None] # [B, H, W, 1, 1]
        depth_repeat_3  = depth.repeat([1, 1, 1, 3, 1]) # [B, H, W, 3, 1]
        base_depth_image = torch.concatenate([depth_repeat_3, base_depth_image], dim=3) # [B, H, W, 4, 1]
        base_depth_image = base_depth_image * self.base_depth_image # [B, H, W, 4, 1]
        
        # B, H, W, 3 * B, H, W, 1

        # B, 1, 1, 4, 4 * B, H, W, 4, 1 -> B, H, W, 4, 1
        pc_3d = torch.matmul(P_inv[:, None, None, ...], base_depth_image) #[B, H, W, 4, 1]

        #print(T[:, None, None, ...].shape)
        pc_3d = torch.matmul(T[:, None, None, ...], pc_3d)[..., 0:3, 0] # [B, H, W, 4, 1] -> [B, H, W, 3]

        if self.is_export_rgb:
            rgb_image = rgb_image.permute(0, 2, 3, 1).contiguous().float() #[B, 3, H, W] -> [B, H, W, 3]
            pc_3d = torch.cat([pc_3d, rgb_image], dim=3) #[B, H, W, 6]

        mask = mask * (depth_image[:, 0] < 300)
        pc_3d = pc_3d.reshape(-1, 6)
        mask = mask.reshape(-1)

        
        return pc_3d, mask.bool()

    def forward(self, image, P, T, P_inv, mask):
        original_image = image.clone()
        image = self.normalize_image(image)
        with torch.no_grad():
            pred_depth, confidence, output_dict = self.meta_arch.inference({'input': image})
            canonical_to_real_scale = P[:, 0, 0, None, None] / 1000.0 # 1000.0 is the focal length of canonical camera
            print(canonical_to_real_scale.shape, pred_depth.shape)
            pred_depth = pred_depth * canonical_to_real_scale # now the depth is metric
        point_cloud, mask = self.depth_image_to_point_cloud_array(pred_depth, P_inv, T, original_image/256.0, mask)
        # pred_depth = torch.nn.functional.interpolate(pred_depth[None, None, :, :], image.shape[:2], mode='bilinear').squeeze()
        return pred_depth, point_cloud, mask

In [5]:
multi_depth_model = Metric3DExportModel(model, is_export_rgb=True)
multi_depth_model.eval()
multi_depth_model.cuda()

B = 1
dummy_image = torch.randn([B, 3, 616, 1064]).cuda()
dummy_T = torch.eye(4).expand(B, -1, -1).cuda()
dummy_P = dummy_T[:, 0:3, :].clone()
dummy_P_inv = torch.eye(4).expand(B, -1, -1).cuda()
dummy_mask = torch.ones([B, 616, 1064]).bool().cuda()
dummy_scale = torch.ones(1).cuda()

In [6]:
with torch.no_grad():
    output = multi_depth_model(dummy_image, dummy_P, dummy_T, dummy_P_inv, dummy_mask)
    print(output[0].shape, output[1].shape, output[2].shape)

torch.Size([1, 1, 1]) torch.Size([1, 1, 616, 1064])
torch.Size([1, 1, 616, 1064]) torch.Size([655424, 6]) torch.Size([655424])


In [7]:
onnx_output = "metric_3d.onnx"
dummy_input = (dummy_image, dummy_P, dummy_T, dummy_P_inv, dummy_mask)
torch.onnx.export(multi_depth_model, dummy_input, onnx_output,
                   input_names=['image', 'P', 'T', 'P_inv', 'mask'],
                   output_names=['pred_depth', 'point_cloud', 'output_mask'], opset_version=11,
                   )

/home/ukenryu/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/ViT_DINO_reg.py:984: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if pad_h == self.patch_size:
/home/ukenryu/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/ViT_DINO_reg.py:986: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if pad_w == self.patch_size:
/home/ukenryu/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/ViT_DINO_reg.py:235: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow

torch.Size([1, 1, 1]) torch.Size([1, 1, 616, 1064])
